Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.

In [ ]:
import os

ROOT_DIR = os.getcwd()
is_colab_mode = False

# 코랩 모드
if ROOT_DIR == "/content":

    from google.colab import drive
    drive.mount('/content/drive')

    is_colab_mode = True


    print("[[ colab ]]")

    # import unicodedata

    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")

    if "raw.tar.gz" not in os.listdir():
        !wget https://github.com/wonbywondev/ML-DL/releases/download/data-v5/raw.tar.gz

    print("· 압축 파일 있음")


    if not os.path.exists(DATA_DIR):
        os.mkdir(DATA_DIR)

    if not os.path.exists(RAW_DIR):
        !tar -xzvf raw.tar.gz -C /content/data

    print("· data 압축 해제 완료")


    # train_json_path = unicodedata.normalize("NFC", train_json_path)
    # val_json_path = unicodedata.normalize("NFC", val_json_path)

# 로컬 모드
else:
    print("[[ local ]]")
    ROOT_DIR = "/".join(ROOT_DIR.split("/")[:-1])
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")



train_edit_json_path = os.path.join(RAW_DIR, "train_original_editorial.json")
train_law_json_path = os.path.join(RAW_DIR, "train_original_news.json")
train_news_json_path = os.path.join(RAW_DIR, "train_original_law.json")
val_edit_json_path = os.path.join(RAW_DIR, "valid_original_editorial.json")
val_law_json_path = os.path.join(RAW_DIR, "valid_original_news.json")
val_news_json_path = os.path.join(RAW_DIR, "valid_original_law.json")

Mounted at /content/drive
[[ colab ]]
--2026-01-20 01:54:15--  https://github.com/wonbywondev/ML-DL/releases/download/data-v5/raw.tar.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1095771901/48b6d68f-5013-4a04-b64a-4f0ba0a78c98?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-20T02%3A49%3A19Z&rscd=attachment%3B+filename%3Draw.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-20T01%3A48%3A56Z&ske=2026-01-20T02%3A49%3A19Z&sks=b&skv=2018-11-09&sig=z9IkB%2BIxlZArhxD8fKGfuWp72ShIGkq9C6bdvf4ql74%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2ODg3NzY1NSwibmJmIjoxNzY4ODc0MDU1LCJwYXRoIjoicmVs

In [ ]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import gc


# 시각화 관련 설정
if not is_colab_mode:
    import matplotlib.font_manager as fm
    try:
        plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
    except:
        try:
            plt.rcParams['font.family'] = 'NanumGothic'
        except:
            plt.rcParams['font.family'] = 'AppleGothic'

    plt.rcParams['axes.unicode_minus'] = False
    fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)

In [ ]:
import json

with open(train_edit_json_path, "r", encoding="utf-8") as f:
    raw_train_edit = json.load(f)

In [ ]:
raw_train_edit

- highlight_indices 항목은 불용어의 인덱스를 표시해둔 리스트이다. 이는 해석 과정에서 유용하게 쓰이는 정보이므로 여기서는 필요가 없다.

In [ ]:
def json_to_dict(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    RESULT_DICT = dict()

    docu_dict_list = raw_data["documents"]

    for result_dict_key, docu_dict in enumerate(docu_dict_list):

        text_dict_list = docu_dict["text"]
        target_index_list = docu_dict["extractive"]
        target = docu_dict["abstractive"][0]

        RESULT_DICT[result_dict_key] = {
            "text": list(),
            "full_text": "",
            "text_index": list(),
            "target_index": target_index_list,
            "target": target
            }

        for paragraph in text_dict_list:
            for sentence_dict in paragraph:
                sentence = sentence_dict["sentence"]
                RESULT_DICT[result_dict_key]["text"].append(sentence)
                RESULT_DICT[result_dict_key]["text_index"].append(sentence_dict["index"])

        RESULT_DICT[result_dict_key]["full_text"] = " ".join(RESULT_DICT[result_dict_key]["text"])

    return RESULT_DICT

In [ ]:
TRAIN_EDIT_DICT = json_to_dict(train_edit_json_path)

In [ ]:
TRAIN_EDIT_DICT[0]

{'text': ['이명박 대통령이 어제 30대 그룹 총수를 모아놓고 "시대적 요구는 역시 총수가 앞장서야 한다. 이미 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 직접 관심을 가져주시면 빨리 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다.',
  "언뜻 보아 무슨 말인지 불분명하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 각각 2000억원과 5000억원을 기부한 사실과 '공생발전'이란 화두를 연결하면 금방 짐작이 간다.",
  '다른 그룹 총수들도 좀 나서라고 은근히 떠민 것이다.',
  '이 대통령은 기부에 대한 후속 선언이 나오지 않은 탓인지 총수들의 사회공헌 방안에 불만을 표시했다는 후문이다.',
  '최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의 자산세를 거둬 약 155조원을 마련하자는 논의가 있었다.',
  '이런 흐름에 한국만 동떨어져 있기는 어려운 게 글로벌 시대의 특징이다.',
  '항간에는 이번 회동 후 삼성을 비롯해 몇몇 그룹이 노블레스 오블리주 방안을 준비하고 있다는 말이 나도는데 대통령의 강요나 포퓰리즘에 의한 압박보다 자발적 문화로 만들어가야 효과가 큰 법이다.',
  '그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고 시간을 줘야 한다.',
  "국가채무 문제로 글로벌 경기 침체 우려가 큰 상황에서 기업들은 '생존'에 큰 부담을 느끼고 있기 때문이다.",
  "이날 전경련에 따르면 30대 그룹은 올해 고용 12만4000명, 투자 114조원 등 '선물'을 준비했다.",
  '세계적인 더블딥이 우려되는 상황에서 공격경영이 어렵겠지만 연초 한 번 발표한 내용을 약간 수정해 내놓은 전경련의 행태는 답답하다.',
  '설립 50주년이 됐으면 좀 더 창의적이고 유연하게 바뀔 때도 됐다.',
  '허창수 전경련 회장은 "대기업ㆍ중소기업이 서로 공생하고 발전할 수 있도록 노력하겠다. 기업이 사회적 책임을 다하겠다"는 원론

### 결측치·중복치

In [ ]:
import pandas as pd

def drop_err(dictionary: dict):

    # 결측치: target_index
    tmp_df = (pd.DataFrame.from_dict(dictionary, orient='index').sort_index()).drop(["text", "text_index"], axis=1)
    target_idx_err_count = sum(tmp_df['target_index'].isna())

    for nan_key in tmp_df[tmp_df["target_index"].isna()].index:
        del dictionary[nan_key]

    tmp_df = tmp_df.drop(["target_index"], axis=1)


    err_target_set = set()
    for key, dict_ in dictionary.items():
        try:
            dict_["target_index"] = sorted(dict_["target_index"])
        except:
            target_idx_err_count += 1
            err_target_set.add(key)


    for err_key in err_target_set:
        del dictionary[err_key]

    print(f"· 결측치(주요 문장 인덱스): {target_idx_err_count}개")


    # 결측치: text_index
    nan_count = 0
    nan_set = set()

    for key, dict_ in dictionary.items():
        for i in range(len(dict_["text_index"])):
            if i not in dict_["text_index"]:
                nan_count += 1
                continue
        del dict_["text_index"]

    for nan_key in nan_set:
        del dictionary[nan_key]


    # 중복치
    for dup_key in tmp_df[tmp_df.duplicated()].index:
        del dictionary[dup_key]


    dictionary = {i: value for i, value in enumerate(dictionary.values())}

    print(f"· 결측치(누락 문장): {nan_count}개")
    print(f"· 중복치 {sum(tmp_df.duplicated())}개")
    print(f"· 수정 후 데이터: {len(dictionary)}개")


    return dictionary

In [ ]:
print("[editorial 학습 데이터]")
TRAIN_EDIT_DICT = drop_err(TRAIN_EDIT_DICT)
print("[editorial 검증 데이터]")
VAL_EDIT_DICT = json_to_dict(val_edit_json_path)
print(f"· 데이터: {len(VAL_EDIT_DICT)}개\n")


print("[law 학습 데이터]")
TRAIN_LAW_DICT = drop_err(json_to_dict(train_law_json_path))
print("[law 검증 데이터]")
VAL_LAW_DICT = json_to_dict(val_law_json_path)
print(f"· 데이터: {len(VAL_LAW_DICT)}개\n")


print("[news 학습 데이터]")
TRAIN_NEWS_DICT = drop_err(json_to_dict(train_news_json_path))
print("[news 검증 데이터]")
VAL_NEWS_DICT = json_to_dict(val_news_json_path)
print(f"· 데이터: {len(VAL_NEWS_DICT)}개")

[editorial 학습 데이터]
· 결측치(주요 문장 인덱스): 0개
· 결측치(누락 문장): 0개
· 중복치 17개
· 수정 후 데이터: 56743개
[editorial 검증 데이터]
· 데이터: 7008개

[law 학습 데이터]
· 결측치(주요 문장 인덱스): 6개
· 결측치(누락 문장): 0개
· 중복치 3개
· 수정 후 데이터: 243974개
[law 검증 데이터]
· 데이터: 30122개

[news 학습 데이터]
· 결측치(주요 문장 인덱스): 0개
· 결측치(누락 문장): 0개
· 중복치 5개
· 수정 후 데이터: 24324개
[news 검증 데이터]
· 데이터: 3004개


In [ ]:
TRAIN_NEWS_DICT[0]

{'text': ['원고가 소속회사의 노동조합에서 분규가 발생하자 노조활동을 구실로 정상적인 근무를 해태하고,',
  '노조조합장이 사임한 경우,',
  '노동조합규약에 동 조합장의 직무를 대행할 자를 규정해 두고 있음에도 원고 자신이 주동하여 노조자치수습대책위원회를 구성하여 그 위원장으로 피선되어 근무시간중에도 노조활동을 벌여 운수업체인 소속회사의 업무에 지장을 초래하고',
  '종업원들에게도 나쁜 영향을 끼쳐 소속회사가 취업규칙을 위반하고',
  '고의로 회사업무능률을 저해하였으며 회사업무상의 지휘명령에 위반하였음을 이유로 원고를 징계해고 하였다면,',
  '이는 원고의 노동조합 활동과는 관계없이 회사취업규칙에 의하여 사내질서를 유지하기 위한 사용자 고유의 징계권에 기하여 이루어진 정당한 징계권의 행사로 보아야 한다.'],
 'full_text': '원고가 소속회사의 노동조합에서 분규가 발생하자 노조활동을 구실로 정상적인 근무를 해태하고, 노조조합장이 사임한 경우, 노동조합규약에 동 조합장의 직무를 대행할 자를 규정해 두고 있음에도 원고 자신이 주동하여 노조자치수습대책위원회를 구성하여 그 위원장으로 피선되어 근무시간중에도 노조활동을 벌여 운수업체인 소속회사의 업무에 지장을 초래하고 종업원들에게도 나쁜 영향을 끼쳐 소속회사가 취업규칙을 위반하고 고의로 회사업무능률을 저해하였으며 회사업무상의 지휘명령에 위반하였음을 이유로 원고를 징계해고 하였다면, 이는 원고의 노동조합 활동과는 관계없이 회사취업규칙에 의하여 사내질서를 유지하기 위한 사용자 고유의 징계권에 기하여 이루어진 정당한 징계권의 행사로 보아야 한다.',
 'target_index': [2, 4, 5],
 'target': '원고가  주동하여 회사업무능률을 저해하고 회사업무상의 지휘명령에 위반하였다면 이에 따른 징계해고는 사내질서를 유지하기 위한 사용자 고유의 정당한 징계권의 행사로 보아야 한다.'}

- law 데이터만 너무 많다. 균형 있게 섞어서 넣는 게 낫지 않을까?

In [ ]:
from datasets import Dataset, DatasetDict

edit_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_EDIT_DICT.values())),
    "val": Dataset.from_list(list(VAL_EDIT_DICT.values()))
})

law_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_LAW_DICT.values())),
    "val": Dataset.from_list(list(VAL_LAW_DICT.values()))
})

news_dataset = DatasetDict({
    "train": Dataset.from_list(list(TRAIN_NEWS_DICT.values())),
    "val": Dataset.from_list(list(VAL_NEWS_DICT.values()))
})

In [ ]:
news_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'full_text', 'target_index', 'target'],
        num_rows: 24324
    })
    val: Dataset({
        features: ['text', 'full_text', 'text_index', 'target_index', 'target'],
        num_rows: 3004
    })
})

# Hugging Face Dataset 변환 및 전처리

In [ ]:
import torch
from transformers import BartForConditionalGeneration
from transformers import PreTrainedTokenizerFast

model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')
tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BartTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


In [ ]:
# def tokenize_function(example):
#     model_inputs = tokenizer(
#         example["full_text"],
#         max_length=512,
#         truncation=True
#     )
#     labels = tokenizer(
#         text_target=example["target"],
#         max_length=256,
#         truncation=True
#     )
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs

In [ ]:
# from transformers import DataCollatorForSeq2Seq
# from transformers import TrainingArguments, Trainer

# tokenized_datasets = news_dataset.map(tokenize_function, batched=True)
# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

# training_args = TrainingArguments(output_dir="test-trainer", report_to="none")

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["val"],
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     )

# trainer.train()

## 실험 모드 통합 및 체크포인트 관리
- 추출/생성 모드를 하나의 설정으로 제어하고, rough-1 평가와 시각화를 동일한 헬퍼로 처리합니다.
- 체크포인트는 Google Drive 경로만 바꿔주면 되도록 기본 변수를 선언했습니다.


In [ ]:
!pip install rouge_score
!pip install evaluate

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=fd5655acfabf0bf524b874208725ca4994cad11feeb5cbfba53ec3cc75273bfd
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
from dataclasses import dataclass, asdict
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
)

EXPERIMENT_DRIVE_ROOT = "/content/drive/MyDrive/data/runs"  # TODO: 환경에 맞는 경로로 수정
# EXPERIMENT_DRIVE_ROOT = "/content/drive/MyDrive/summarization_runs"  # TODO: 환경에 맞는 경로로 수정
Path(EXPERIMENT_DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

@dataclass
class ExperimentConfig:
    mode: str = "baseline"          # baseline | extractive | hybrid
    dataset: str = "news"           # edit | law | news | all
    experiment_name: str = "news_baseline"
    drive_root: str = EXPERIMENT_DRIVE_ROOT
    resume: bool = True
    include_extractive: bool = False
    use_predicted_extractive: bool = True
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 1
    learning_rate: float = 5e-5
    generation_max_length: int = 128
    save_total_limit: int = 2
    extractive_model_name: str = "klue/bert-base"
    generative_model_name: str = "gogamza/kobart-summarization"
    positive_class_weight: float = 2.0
    max_pos_oversample: int = 2

AVAILABLE_DATASET_DICTS: Dict[str, DatasetDict] = {
    "edit": edit_dataset,
    "law": law_dataset,
    "news": news_dataset,
}
AVAILABLE_DATASET_DICTS["all"] = DatasetDict({
    split: concatenate_datasets([
        edit_dataset[split],
        law_dataset[split],
        news_dataset[split],
    ])
    for split in edit_dataset.keys()
})

PREDICTED_EXTRACTIVE_DATASETS: Dict[str, DatasetDict] = {}

rouge_metric = evaluate.load("rouge")

import inspect





In [ ]:
def resolve_experiment_dir(config: ExperimentConfig, sub_dir: str = "") -> Path:
    base_dir = Path(config.drive_root) / config.experiment_name
    if sub_dir:
        base_dir = base_dir / sub_dir
    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir


def latest_checkpoint_path(output_dir: Path) -> Optional[str]:
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
        )
    return str(checkpoints[-1]) if checkpoints else None


def log_metrics_to_csv(save_path: Path, metrics: Dict):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    row = {**metrics, "timestamp": datetime.now().isoformat()}
    df = pd.DataFrame([row])
    if save_path.exists():
        existing = pd.read_csv(save_path)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(save_path, index=False)


def add_extractive_fields(example: Dict) -> Dict:
    target_index = example.get("target_index") or []
    sentences = example.get("text") or []
    chosen = [sentences[i] for i in target_index if i < len(sentences)]
    if not chosen and sentences:
        chosen = sentences[:1]
    return {
        "extract_summary": " ".join(chosen),
        "extract_sentence_count": len(chosen),
        }


def prepare_dataset(config: ExperimentConfig, add_extractive: bool = False) -> DatasetDict:
    dataset = AVAILABLE_DATASET_DICTS[config.dataset]
    if add_extractive:
        dataset = dataset.map(
            add_extractive_fields,
            desc=f"[{config.mode}] 추출 요약 필드 생성 ({config.dataset})",
            )
    return dataset



In [ ]:
def build_generative_tokenizer(config: ExperimentConfig):
    from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast
    model = BartForConditionalGeneration.from_pretrained(config.generative_model_name)
    tok = PreTrainedTokenizerFast.from_pretrained(config.generative_model_name)
    return model, tok

def tokenize_with_mode(examples, tokenizer, mode: str):
    sources = []
    texts = examples["full_text"]
    predicted_summaries = examples.get("predicted_extract_summary")
    fallback_summaries = examples.get("extract_summary")
    for idx, text in enumerate(texts):
        source_text = text
        if mode == "hybrid":
            extract = None
            if predicted_summaries is not None:
                extract = predicted_summaries[idx]
            if (extract is None or extract == "") and fallback_summaries is not None:
                extract = fallback_summaries[idx]
            if extract:
                source_text = f"[추출요약] {extract} </s> [본문] {text}"
        sources.append(source_text)
    model_inputs = tokenizer(sources, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples["target"], max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def tokenize_generative_dataset(dataset: DatasetDict, config: ExperimentConfig, tokenizer) -> DatasetDict:
    remove_columns = dataset["train"].column_names
    return DatasetDict({
        split: ds.map(
            lambda batch: tokenize_with_mode(batch, tokenizer, config.mode),
            batched=True,
            remove_columns=remove_columns,
            desc=f"[{config.mode}] 생성 토크나이징 ({config.dataset}-{split})",
        )
        for split, ds in dataset.items()
    })

def build_rouge_fn(tokenizer):
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        rouge = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            rouge_types=["rouge1", "rouge2", "rougeL"],
            use_stemmer=True,
        )
        return {"rough1": rouge["rouge1"], "rouge2": rouge["rouge2"], "rougeL": rouge["rougeL"]}
    return compute_metrics

def plot_training_history(trainer, save_path: Path):
    history_df = pd.DataFrame(trainer.state.log_history)
    if history_df.empty or "loss" not in history_df.columns:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    history_df = history_df.dropna(subset=["loss"])
    ax.plot(history_df["step"], history_df["loss"], label="loss")
    if "eval_loss" in history_df.columns:
        eval_df = history_df.dropna(subset=["eval_loss"])
        if not eval_df.empty:
            ax.plot(eval_df["step"], eval_df["eval_loss"], label="eval_loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=200); plt.close(fig)

def preview_generation_samples(trainer, tokenizer, raw_dataset: DatasetDict, tokenized_dataset: DatasetDict, split: str = "val", num_samples: int = 3):
    sample_count = min(num_samples, len(raw_dataset[split]))
    indices = list(range(sample_count))
    raw_subset = raw_dataset[split].select(indices)
    tokenized_subset = tokenized_dataset[split].select(indices)
    generations = trainer.predict(tokenized_subset).predictions
    if isinstance(generations, tuple):
        generations = generations[0]
    decoded_preds = tokenizer.batch_decode(generations, skip_special_tokens=True)
    preview = []
    for idx in range(sample_count):
        preview.append({
            "full_text": raw_subset[idx]["full_text"][:200] + "...",
            "target": raw_subset[idx]["target"],
            "prediction": decoded_preds[idx],
        })
    return pd.DataFrame(preview)



In [ ]:
def tokenize_sentence_batch(batch, tokenizer):
    return tokenizer(batch["sentence"], truncation=True, padding=True, max_length=256)



In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, device=self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is None or labels is None:
            loss = outputs.get("loss")
        else:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss



In [ ]:
def build_predicted_extractive_dataset(raw_dataset: DatasetDict, tokenized_sentence_ds: DatasetDict, trainer) -> DatasetDict:
    predicted_splits = {}
    for split, split_dataset in raw_dataset.items():
        sentence_split = tokenized_sentence_ds[split] if split in tokenized_sentence_ds else None
        if sentence_split is None or len(sentence_split) == 0:
            predicted_splits[split] = split_dataset
            continue
        predictions = trainer.predict(sentence_split)
        logits = predictions.predictions
        if logits.ndim == 1:
            pos_scores = logits
        else:
            pos_scores = logits[:, -1]
        pred_labels = logits.argmax(axis=-1)
        doc_ids = sentence_split["doc_id"]
        sent_indices = sentence_split["sentence_index"]
        doc_positive_indices = defaultdict(list)
        doc_scores = defaultdict(list)
        for doc_id, sent_idx, label, score in zip(doc_ids, sent_indices, pred_labels, pos_scores):
            doc_scores[int(doc_id)].append((float(score), int(sent_idx)))
            if int(label) == 1:
                doc_positive_indices[int(doc_id)].append(int(sent_idx))

        new_examples = []
        for doc_id, example in enumerate(split_dataset):
            sentences = example.get("text") or []
            chosen_indices = sorted(set(doc_positive_indices.get(doc_id, [])))
            if not chosen_indices and doc_scores.get(doc_id):
                best_idx = max(doc_scores[doc_id], key=lambda x: x[0])[1]
                chosen_indices = [best_idx]
            elif not chosen_indices and sentences:
                chosen_indices = [0]

            summary = " ".join(
                sentences[idx] for idx in chosen_indices if 0 <= idx < len(sentences)
            ) if sentences and chosen_indices else ""

            enriched = dict(example)
            enriched["predicted_extract_indices"] = chosen_indices
            enriched["predicted_extract_summary"] = summary
            enriched["predicted_sentence_count"] = len(chosen_indices)
            new_examples.append(enriched)

        predicted_splits[split] = Dataset.from_list(new_examples)
    return DatasetDict(predicted_splits)


def preview_extractive_predictions(predicted_dataset: DatasetDict, split: str = "val", num_samples: int = 3):
    if split not in predicted_dataset:
        return pd.DataFrame()
    max_count = min(num_samples, len(predicted_dataset[split]))
    rows = []
    for idx in range(max_count):
        sample = predicted_dataset[split][idx]
        rows.append({
            "predicted_indices": sample.get("predicted_extract_indices", []),
            "target_indices": sample.get("target_index", []),
            "predicted_summary": (sample.get("predicted_extract_summary") or "")[:200] + "...",
        })
    return pd.DataFrame(rows)



In [ ]:
def train_extractive_model(config: ExperimentConfig) -> Dict:
    dataset = prepare_dataset(config, add_extractive=True)
    sentence_ds = DatasetDict({
        split: rebalance_sentences(build_sentence_level_dataset(dataset[split]), max_pos_oversample=config.max_pos_oversample)
        for split in dataset.keys()
    })

    tokenizer = AutoTokenizer.from_pretrained(config.extractive_model_name)
    tokenized = DatasetDict({
        split: sentence_ds[split].map(
            lambda batch: tokenize_sentence_batch(batch, tokenizer),
            batched=True,
            remove_columns=["sentence"],
            desc=f"[{config.mode}] 문장 토크나이징 ({split})",
        )
        for split in sentence_ds.keys()
    })

    data_collator = DataCollatorWithPadding(tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.extractive_model_name,
        num_labels=2,
    )

    output_dir = resolve_experiment_dir(config, "extractive")

    sig = inspect.signature(TrainingArguments.__init__)
    training_kwargs = {
        "output_dir": str(output_dir),
        "learning_rate": config.learning_rate,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "per_device_eval_batch_size": config.per_device_train_batch_size,
        "num_train_epochs": config.num_train_epochs,
    }

    def add_kwarg(names, value):
        if isinstance(names, str):
            names = [names]
        for name in names:
            if name in sig.parameters:
                training_kwargs[name] = value
                return

    add_kwarg("gradient_accumulation_steps", config.gradient_accumulation_steps)
    add_kwarg(("evaluation_strategy", "eval_strategy"), "steps")
    add_kwarg("save_strategy", "steps")
    add_kwarg("logging_steps", 50)
    add_kwarg("save_steps", 200)
    add_kwarg("eval_steps", 200)
    add_kwarg("load_best_model_at_end", True)
    add_kwarg("metric_for_best_model", "f1")
    add_kwarg("greater_is_better", True)
    add_kwarg("save_total_limit", config.save_total_limit)
    add_kwarg("report_to", ["none"])

    if (
        "evaluation_strategy" not in training_kwargs
        and "eval_strategy" not in training_kwargs
        and "evaluate_during_training" in sig.parameters
    ):
        training_kwargs["evaluate_during_training"] = True

    training_args = TrainingArguments(**training_kwargs)

    neg_weight = 1.0
    pos_weight = config.positive_class_weight
    class_weights = [neg_weight, pos_weight]

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=training_args,
        train_dataset=tokenized.get("train"),
        eval_dataset=tokenized.get("val"),
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_extractive_metrics,
    )

    resume_ckpt = latest_checkpoint_path(output_dir) if config.resume else None
    trainer.train(resume_from_checkpoint=resume_ckpt)

    eval_metrics = trainer.evaluate()
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)

    predicted_dataset = build_predicted_extractive_dataset(dataset, tokenized, trainer)
    PREDICTED_EXTRACTIVE_DATASETS[config.experiment_name] = predicted_dataset
    preview_df = preview_extractive_predictions(predicted_dataset)

    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "datasets": sentence_ds,
        "tokenized": tokenized,
        "metrics": eval_metrics,
        "predicted_dataset": predicted_dataset,
        "preview": preview_df,
    }







In [ ]:
def preview_generation_samples(trainer, tokenizer, raw_dataset: DatasetDict, tokenized_dataset: DatasetDict, split: str = "val", num_samples: int = 3):
    sample_count = min(num_samples, len(raw_dataset[split]))
    sample_indices = list(range(sample_count))
    raw_subset = raw_dataset[split].select(sample_indices)
    tokenized_subset = tokenized_dataset[split].select(sample_indices)
    generations = trainer.predict(tokenized_subset).predictions
    if isinstance(generations, tuple):
        generations = generations[0]
    decoded_preds = tokenizer.batch_decode(generations, skip_special_tokens=True)
    preview = []
    for idx in range(sample_count):
        preview.append({
            "full_text": raw_subset[idx]["full_text"][:200] + "...",
            "target": raw_subset[idx]["target"],
            "prediction": decoded_preds[idx],
        })
    return pd.DataFrame(preview)



In [ ]:
def run_generative_experiment(config: ExperimentConfig) -> Dict:
    dataset = None
    dataset_source = "original"
    if config.include_extractive and config.use_predicted_extractive:
        cached = PREDICTED_EXTRACTIVE_DATASETS.get(config.experiment_name)
        if cached is not None:
            dataset = cached
            dataset_source = "predicted"

    if dataset is None:
        add_extractive = config.include_extractive or config.mode in {"extractive", "hybrid"}
        dataset = prepare_dataset(config, add_extractive=add_extractive)

    model, gen_tokenizer = build_generative_tokenizer(config)
    tokenized = tokenize_generative_dataset(dataset, config, gen_tokenizer)

    output_dir = resolve_experiment_dir(config, f"generative_{config.mode}")

    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    training_kwargs = {
        "output_dir": str(output_dir),
        "learning_rate": config.learning_rate,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "per_device_eval_batch_size": config.per_device_train_batch_size,
        "num_train_epochs": config.num_train_epochs,
    }

    def add_kwarg(names, value):
        if isinstance(names, str):
            names = [names]
        for name in names:
            if name in sig.parameters:
                training_kwargs[name] = value
                return

    add_kwarg("gradient_accumulation_steps", config.gradient_accumulation_steps)
    add_kwarg("predict_with_generate", True)
    add_kwarg("generation_max_length", config.generation_max_length)
    add_kwarg(("evaluation_strategy", "eval_strategy"), "steps")
    add_kwarg("save_strategy", "steps")
    add_kwarg("logging_steps", 50)
    add_kwarg("save_steps", 200)
    add_kwarg("eval_steps", 200)
    add_kwarg("save_total_limit", config.save_total_limit)
    add_kwarg("report_to", ["none"])

    if (
        "evaluation_strategy" not in training_kwargs
        and "eval_strategy" not in training_kwargs
        and "evaluate_during_training" in sig.parameters
    ):
        training_kwargs["evaluate_during_training"] = True

    training_args = Seq2SeqTrainingArguments(**training_kwargs)

    data_collator = DataCollatorForSeq2Seq(tokenizer=gen_tokenizer, model=model)
    compute_metrics = build_rouge_fn(gen_tokenizer)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized.get("train"),
        eval_dataset=tokenized.get("val"),
        tokenizer=gen_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    resume_ckpt = latest_checkpoint_path(output_dir) if config.resume else None
    trainer.train(resume_from_checkpoint=resume_ckpt)

    eval_metrics = trainer.evaluate(max_length=config.generation_max_length)
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)
    plot_training_history(trainer, output_dir / "history.png")

    return {
        "trainer": trainer,
        "tokenizer": gen_tokenizer,
        "dataset": dataset,
        "dataset_source": dataset_source,
        "tokenized": tokenized,
        "metrics": eval_metrics,
        "preview": preview_generation_samples(trainer, gen_tokenizer, dataset, tokenized),
    }



In [ ]:
baseline_config = ExperimentConfig(
    mode="baseline",
    dataset="news",
    experiment_name="news_baseline",
    num_train_epochs=1,
)

baseline_run = run_generative_experiment(baseline_config)
baseline_run["preview"]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BartTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


[baseline] 생성 토크나이징 (news-train):   0%|          | 0/24324 [00:00<?, ? examples/s]

[baseline] 생성 토크나이징 (news-val):   0%|          | 0/3004 [00:00<?, ? examples/s]

/tmp/ipython-input-1418177250.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss


,full_text,target,prediction
0,"[1] 취소소송은 처분 등이 있음을 안 날부터 90일 이내에 제기하여야 하고, 처분...","취소소송은 처분 등이 있다는 것을 안 때로부터 90일 이내에 제기하여야 하고, 행정...","취소소송은 처분 등이 있음을 안 날부터 90일 이내에 제기하여야 하고, 행정소송법 ..."
1,[1] 항고소송의 대상이 되는 행정처분이라 함은 원칙적으로 행정청의 공법상 행위로서...,항고소송의 대상이 되는 행정처분이란 일반 국민의 권리의무에 직접 영향을 미치는 행위...,항고소송의 대상이 되는 행정처분이라 함은 원칙적으로 행정청의 공법상 행위로서 특정 ...
2,취득세는 본래 재화의 이전이라는 사실 자체를 포착하여 거기에 담세력을 인정하고 부과...,"취득세는 사실상의 취득행위 자체를 과세객체로 하고, 지방세법에 따르면 부동산 취득에...",취득세는 취득자가 실질적으로 완전한 내용의 소유권을 취득하는가의 여부에 관계없이 사...


In [ ]:
hybrid_config = ExperimentConfig(
    mode="hybrid",
    dataset="news",
    experiment_name="news_hybrid",
    include_extractive=True,
    num_train_epochs=1,
)

extractive_run = train_extractive_model(hybrid_config)
extractive_run["preview"]

[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/24324 [00:00<?, ? examples/s]

[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/3004 [00:00<?, ? examples/s]

[hybrid] 문장 토크나이징 (train):   0%|          | 0/314683 [00:00<?, ? examples/s]

[hybrid] 문장 토크나이징 (val):   0%|          | 0/23454 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-927717850.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
600,0.687100,0.950776,0.384241,0.384241,1.000000,0.555165
800,0.758500,0.800296,0.384241,0.384241,1.000000,0.555165
1000,0.673700,0.789671,0.384241,0.384241,1.000000,0.555165


In [ ]:
hybrid_run = run_generative_experiment(hybrid_config)
hybrid_run["preview"]

In [ ]:
# [Improved Hybrid] Separated Training Pipeline with Robust Checkpointing
from transformers import TrainerCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import Dataset, DatasetDict
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, DataCollatorForSeq2Seq
import os
from pathlib import Path

# --- Utility Functions (Redefined for Safety) ---

def resolve_experiment_dir(config: ExperimentConfig, sub_dir: str = "") -> Path:
    base_dir = Path(config.drive_root) / config.experiment_name
    if sub_dir:
        base_dir = base_dir / sub_dir
    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir

def latest_checkpoint_path(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if checkpoints:
        return str(checkpoints[-1])
    return None

def build_sentence_level_dataset(dataset):
    sentence_examples = []
    for doc_id, example in enumerate(dataset):
        sentences = example.get("text", [])
        target_indices = set(example.get("target_index", []))
        for idx, sent in enumerate(sentences):
            label = 1 if idx in target_indices else 0
            sentence_examples.append({
                "sentence": sent,
                "label": label,
                "doc_id": doc_id,
                "sentence_index": idx
            })
    return Dataset.from_list(sentence_examples)

def rebalance_sentences(sentence_dataset: Dataset, max_pos_oversample: int = 2) -> Dataset:
    df = sentence_dataset.to_pandas()
    positives = df[df["label"] == 1]
    negatives = df[df["label"] == 0]
    if positives.empty or negatives.empty:
        return sentence_dataset
    pos_count = len(positives)
    neg_count = len(negatives)
    oversample_factor = min(max_pos_oversample, max(1, neg_count // max(pos_count, 1)))
    positives_oversampled = pd.concat([positives] * oversample_factor, ignore_index=True)
    target_neg_count = oversample_factor * len(positives)
    sampled_negatives = negatives.sample(n=min(target_neg_count, len(negatives)), random_state=42, replace=False)
    balanced_df = pd.concat([positives_oversampled, sampled_negatives], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return Dataset.from_pandas(balanced_df, preserve_index=False)

# --- Extractive Components ---

class RobustWeightedTrainer(Trainer):
    def __init__(self, pos_weight_value=1.0, **kwargs):
        super().__init__(**kwargs)
        self.pos_weight = torch.tensor([pos_weight_value], device=self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, self.pos_weight.item()], device=model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss = outputs.get("loss")
        return (loss, outputs) if return_outputs else loss

def compute_extractive_metrics_robust(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

def build_robust_predicted_dataset(raw_dataset: DatasetDict, tokenized_sentence_ds: DatasetDict, trainer, top_k: int = 3) -> DatasetDict:
    predicted_splits = {}
    for split, split_dataset in raw_dataset.items():
        if split not in tokenized_sentence_ds:
            predicted_splits[split] = split_dataset
            continue
        sentence_split = tokenized_sentence_ds[split]
        if len(sentence_split) == 0:
            predicted_splits[split] = split_dataset
            continue

        print(f"[{split}] 예측 수행 중...")
        predictions = trainer.predict(sentence_split)
        logits = predictions.predictions
        pos_scores = logits[:, 1] if logits.ndim == 2 else logits

        doc_ids = sentence_split["doc_id"]
        sent_indices = sentence_split["sentence_index"]

        doc_scores = defaultdict(list)
        for d_id, s_idx, score in zip(doc_ids, sent_indices, pos_scores):
            doc_scores[int(d_id)].append((float(score), int(s_idx)))

        new_examples = []
        for doc_id, example in enumerate(split_dataset):
            sentences = example.get("text", [])
            scores = doc_scores.get(doc_id, [])
            scores.sort(key=lambda x: x[0], reverse=True)
            top_k_indices = sorted([idx for _, idx in scores[:top_k]])

            if not top_k_indices and sentences:
                top_k_indices = list(range(min(len(sentences), top_k)))

            summary = " ".join([sentences[idx] for idx in top_k_indices if 0 <= idx < len(sentences)])

            enriched = dict(example)
            enriched["predicted_extract_indices"] = top_k_indices
            enriched["predicted_extract_summary"] = summary
            enriched["predicted_sentence_count"] = len(top_k_indices)
            new_examples.append(enriched)

        predicted_splits[split] = Dataset.from_list(new_examples)
        print(f"[{split}] Top-{top_k} 기반 추출 요약 생성 완료")

    return DatasetDict(predicted_splits)

# --- Execution Functions ---

def train_robust_extractive(config: ExperimentConfig):
    print("=== [Step 1] Robust Extractive Model 학습 시작 ===")
    dataset = prepare_dataset(config, add_extractive=True)

    sentence_ds = DatasetDict({
        split: rebalance_sentences(build_sentence_level_dataset(dataset[split]), max_pos_oversample=config.max_pos_oversample)
        for split in dataset.keys()
    })

    tokenizer = AutoTokenizer.from_pretrained(config.extractive_model_name)
    tokenized = DatasetDict({
        split: sentence_ds[split].map(
            lambda batch: tokenize_sentence_batch(batch, tokenizer),
            batched=True,
            remove_columns=["sentence"],
            desc=f"[Extractive] 문장 토크나이징 ({split})",
        )
        for split in sentence_ds.keys()
    })

    model = AutoModelForSequenceClassification.from_pretrained(config.extractive_model_name, num_labels=2)
    robust_weight = max(config.positive_class_weight, 4.0)

    output_dir = resolve_experiment_dir(config, "robust_extractive")

    trainer = RobustWeightedTrainer(
        pos_weight_value=robust_weight,
        model=model,
        args=TrainingArguments(
            output_dir=str(output_dir),
            learning_rate=config.learning_rate,
            per_device_train_batch_size=config.per_device_train_batch_size,
            per_device_eval_batch_size=config.per_device_train_batch_size,
            num_train_epochs=config.num_train_epochs,
            eval_strategy="steps",
            eval_steps=100,
            save_steps=100,
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            save_total_limit=2,
            report_to=["none"],
            remove_unused_columns=True
        ),
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["val"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_extractive_metrics_robust,
    )

    # Checkpoint Loading
    resume_ckpt = None
    if config.resume:
        resume_ckpt = latest_checkpoint_path(output_dir)
        if resume_ckpt:
            print(f">>> [Extractive] 체크포인트 발견: {resume_ckpt}. 학습을 재개합니다.")
        else:
            print(">>> [Extractive] 체크포인트 없음. 처음부터 학습을 시작합니다.")

    trainer.train(resume_from_checkpoint=resume_ckpt)
    metrics = trainer.evaluate()
    print(">>> [Extractive] 최종 평가 결과:", metrics)

    predicted_dataset = build_robust_predicted_dataset(dataset, tokenized, trainer, top_k=3)
    PREDICTED_EXTRACTIVE_DATASETS[config.experiment_name] = predicted_dataset
    print(">>> [Extractive] 예측 데이터셋 메모리 저장 완료.")

    return trainer, metrics, predicted_dataset

def train_robust_generative(config: ExperimentConfig):
    print("=== [Step 2] Robust Generative Model 학습 시작 ===")

    # 1. Dataset Preparation
    dataset = None
    dataset_source = "original"
    if config.include_extractive and config.use_predicted_extractive:
        cached = PREDICTED_EXTRACTIVE_DATASETS.get(config.experiment_name)
        if cached is not None:
            dataset = cached
            dataset_source = "predicted"
            print(">>> [Generative] 예측된 추출 요약 데이터(Predicted)를 사용합니다.")
        else:
            print(">>> [Generative] Warning: 예측된 데이터를 찾을 수 없어 원본 데이터를 사용합니다.")
            dataset = prepare_dataset(config, add_extractive=True)
    else:
        dataset = prepare_dataset(config, add_extractive=False)

    # 2. Tokenization
    model, gen_tokenizer = build_generative_tokenizer(config)
    tokenized = tokenize_generative_dataset(dataset, config, gen_tokenizer)

    # 3. Trainer Setup
    output_dir = resolve_experiment_dir(config, f"generative_{config.mode}")

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_train_batch_size,
        num_train_epochs=config.num_train_epochs,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        predict_with_generate=True,
        generation_max_length=config.generation_max_length,
        eval_strategy="steps",
        save_strategy="steps",
        logging_steps=50,
        save_steps=200,
        eval_steps=200,
        save_total_limit=config.save_total_limit,
        report_to=["none"]
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=gen_tokenizer, model=model)
    compute_metrics = build_rouge_fn(gen_tokenizer)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized.get("train"),
        eval_dataset=tokenized.get("val"),
        tokenizer=gen_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # 4. Checkpoint Loading (Explicit)
    resume_ckpt = None
    if config.resume:
        resume_ckpt = latest_checkpoint_path(output_dir)
        if resume_ckpt:
            print(f">>> [Generative] 체크포인트 발견: {resume_ckpt}. 학습을 재개합니다.")
        else:
            print(">>> [Generative] 체크포인트 없음. 처음부터 학습을 시작합니다.")

    trainer.train(resume_from_checkpoint=resume_ckpt)

    # 5. Evaluation
    eval_metrics = trainer.evaluate(max_length=config.generation_max_length)
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)
    plot_training_history(trainer, output_dir / "history.png")

    print(">>> [Generative] 최종 평가 결과:", eval_metrics)

    return {
        "trainer": trainer,
        "tokenizer": gen_tokenizer,
        "dataset": dataset,
        "metrics": eval_metrics,
        "preview": preview_generation_samples(trainer, gen_tokenizer, dataset, tokenized),
    }

# --- Execution Config ---
robust_hybrid_config = ExperimentConfig(
    mode="hybrid",
    dataset="news",
    experiment_name="news_robust_hybrid",
    num_train_epochs=1,
    positive_class_weight=4.0
)

# --- 실행 명령 ---
# 1. 추출 요약 학습
ext_result = train_robust_extractive(robust_hybrid_config)

=== [Step 1] Robust Extractive Model 학습 시작 ===


[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/24324 [00:00<?, ? examples/s]

[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/3004 [00:00<?, ? examples/s]

[Extractive] 문장 토크나이징 (train):   0%|          | 0/291880 [00:00<?, ? examples/s]

[Extractive] 문장 토크나이징 (val):   0%|          | 0/18024 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1620790157.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `RobustWeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


>>> [Extractive] 체크포인트 발견: /content/drive/MyDrive/data/runs/news_robust_hybrid/robust_extractive/checkpoint-3500. 학습을 재개합니다.


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
3600,0.829800,0.898274,0.649024,0.659463,0.616289,0.637146
3700,0.618500,0.637260,0.500000,0.500000,1.000000,0.666667
3800,0.691400,0.627257,0.657124,0.646311,0.694075,0.669342
3900,0.603400,0.672524,0.658233,0.633671,0.750111,0.686992
4000,0.557600,0.636801,0.660397,0.619849,0.829561,0.709534
4100,0.616200,0.668437,0.655626,0.612914,0.844763,0.710400
4200,0.916000,1.002317,0.532346,0.789474,0.088216,0.158698
4300,0.682100,0.696309,0.500000,0.500000,1.000000,0.666667
4400,0.743800,0.661838,0.500000,0.500000,1.000000,0.666667
4500,0.720200,0.721262,0.500000,0.500000,1.000000,0.666667


KeyboardInterrupt: 

In [31]:
# [Improved Hybrid] Separated Training Pipeline with Resume/Skip Capability
from transformers import TrainerCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import Dataset, DatasetDict
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, DataCollatorForSeq2Seq
import os
from pathlib import Path

# --- Utility Functions ---

def resolve_experiment_dir(config: ExperimentConfig, sub_dir: str = "") -> Path:
    base_dir = Path(config.drive_root) / config.experiment_name
    if sub_dir:
        base_dir = base_dir / sub_dir
    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir

def latest_checkpoint_path(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if checkpoints:
        return str(checkpoints[-1])
    return None

def build_sentence_level_dataset(dataset):
    sentence_examples = []
    for doc_id, example in enumerate(dataset):
        sentences = example.get("text", [])
        target_indices = set(example.get("target_index", []))
        for idx, sent in enumerate(sentences):
            label = 1 if idx in target_indices else 0
            sentence_examples.append({
                "sentence": sent,
                "label": label,
                "doc_id": doc_id,
                "sentence_index": idx
            })
    return Dataset.from_list(sentence_examples)

def rebalance_sentences(sentence_dataset: Dataset, max_pos_oversample: int = 2) -> Dataset:
    df = sentence_dataset.to_pandas()
    positives = df[df["label"] == 1]
    negatives = df[df["label"] == 0]
    if positives.empty or negatives.empty:
        return sentence_dataset
    pos_count = len(positives)
    neg_count = len(negatives)
    oversample_factor = min(max_pos_oversample, max(1, neg_count // max(pos_count, 1)))
    positives_oversampled = pd.concat([positives] * oversample_factor, ignore_index=True)
    target_neg_count = oversample_factor * len(positives)
    sampled_negatives = negatives.sample(n=min(target_neg_count, len(negatives)), random_state=42, replace=False)
    balanced_df = pd.concat([positives_oversampled, sampled_negatives], ignore_index=True)
    balanced_df = balanced_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return Dataset.from_pandas(balanced_df, preserve_index=False)

# --- Extractive Components ---

class RobustWeightedTrainer(Trainer):
    def __init__(self, pos_weight_value=1.0, **kwargs):
        super().__init__(**kwargs)
        self.pos_weight = torch.tensor([pos_weight_value], device=self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, self.pos_weight.item()], device=model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss = outputs.get("loss")
        return (loss, outputs) if return_outputs else loss

def compute_extractive_metrics_robust(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

def build_robust_predicted_dataset(raw_dataset: DatasetDict, tokenized_sentence_ds: DatasetDict, trainer, top_k: int = 3) -> DatasetDict:
    predicted_splits = {}
    for split, split_dataset in raw_dataset.items():
        if split not in tokenized_sentence_ds:
            predicted_splits[split] = split_dataset
            continue
        sentence_split = tokenized_sentence_ds[split]
        if len(sentence_split) == 0:
            predicted_splits[split] = split_dataset
            continue

        print(f"[{split}] 예측 수행 중...")
        predictions = trainer.predict(sentence_split)
        logits = predictions.predictions
        pos_scores = logits[:, 1] if logits.ndim == 2 else logits

        doc_ids = sentence_split["doc_id"]
        sent_indices = sentence_split["sentence_index"]

        doc_scores = defaultdict(list)
        for d_id, s_idx, score in zip(doc_ids, sent_indices, pos_scores):
            doc_scores[int(d_id)].append((float(score), int(s_idx)))

        new_examples = []
        for doc_id, example in enumerate(split_dataset):
            sentences = example.get("text", [])
            scores = doc_scores.get(doc_id, [])
            scores.sort(key=lambda x: x[0], reverse=True)
            top_k_indices = sorted([idx for _, idx in scores[:top_k]])
            if not top_k_indices and sentences:
                top_k_indices = list(range(min(len(sentences), top_k)))
            summary = " ".join([sentences[idx] for idx in top_k_indices if 0 <= idx < len(sentences)])
            enriched = dict(example)
            enriched["predicted_extract_indices"] = top_k_indices
            enriched["predicted_extract_summary"] = summary
            enriched["predicted_sentence_count"] = len(top_k_indices)
            new_examples.append(enriched)

        predicted_splits[split] = Dataset.from_list(new_examples)
        print(f"[{split}] Top-{top_k} 기반 추출 요약 생성 완료")

    return DatasetDict(predicted_splits)

# --- Execution Functions ---

def train_robust_extractive(config: ExperimentConfig):
    # (Original training function)
    print("=== [Step 1] Robust Extractive Model 학습 시작 ===")
    dataset = prepare_dataset(config, add_extractive=True)
    sentence_ds = DatasetDict({
        split: rebalance_sentences(build_sentence_level_dataset(dataset[split]), max_pos_oversample=config.max_pos_oversample)
        for split in dataset.keys()
    })
    tokenizer = AutoTokenizer.from_pretrained(config.extractive_model_name)
    tokenized = DatasetDict({
        split: sentence_ds[split].map(lambda batch: tokenize_sentence_batch(batch, tokenizer), batched=True, remove_columns=["sentence"], desc=f"[Extractive] 문장 토크나이징 ({split})")
        for split in sentence_ds.keys()
    })
    model = AutoModelForSequenceClassification.from_pretrained(config.extractive_model_name, num_labels=2)
    robust_weight = max(config.positive_class_weight, 4.0)
    output_dir = resolve_experiment_dir(config, "robust_extractive")
    trainer = RobustWeightedTrainer(
        pos_weight_value=robust_weight, model=model,
        args=TrainingArguments(
            output_dir=str(output_dir), learning_rate=config.learning_rate, per_device_train_batch_size=config.per_device_train_batch_size, per_device_eval_batch_size=config.per_device_train_batch_size, num_train_epochs=config.num_train_epochs, eval_strategy="steps", eval_steps=100, save_steps=100, logging_steps=50, load_best_model_at_end=True, metric_for_best_model="f1", save_total_limit=2, report_to=["none"], remove_unused_columns=True
        ),
        train_dataset=tokenized["train"], eval_dataset=tokenized["val"], tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_extractive_metrics_robust,
    )

    resume_ckpt = None
    if config.resume:
        resume_ckpt = latest_checkpoint_path(output_dir)
        if resume_ckpt: print(f">>> [Extractive] 체크포인트 발견: {resume_ckpt}. 학습을 재개합니다.")
        else: print(">>> [Extractive] 체크포인트 없음. 처음부터 학습을 시작합니다.")

    trainer.train(resume_from_checkpoint=resume_ckpt)
    metrics = trainer.evaluate()
    print(">>> [Extractive] 최종 평가 결과:", metrics)

    predicted_dataset = build_robust_predicted_dataset(dataset, tokenized, trainer, top_k=3)
    PREDICTED_EXTRACTIVE_DATASETS[config.experiment_name] = predicted_dataset
    print(">>> [Extractive] 예측 데이터셋 메모리 저장 완료.")
    return trainer, metrics, predicted_dataset

def eval_and_continue_extractive(config: ExperimentConfig):
    """
    학습된 Extractive 모델 체크포인트를 로드하여 평가하고, 다음 단계(Generative)를 위한 예측 데이터를 생성합니다.
    학습(Train) 과정은 건너뜁니다.
    """
    print("=== [Step 1.5] 중단된 Extractive 모델 평가 및 예측 데이터 생성 (Skip Training) ===")
    output_dir = resolve_experiment_dir(config, "robust_extractive")
    ckpt_path = latest_checkpoint_path(output_dir)

    if not ckpt_path:
        raise FileNotFoundError(f"체크포인트를 찾을 수 없습니다: {output_dir}")

    print(f">>> 체크포인트에서 모델 로드 중: {ckpt_path}")

    # 1. Load Model & Tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(ckpt_path, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)

    # 2. Prepare Data
    dataset = prepare_dataset(config, add_extractive=True)
    sentence_ds = DatasetDict({
        split: rebalance_sentences(build_sentence_level_dataset(dataset[split]), max_pos_oversample=config.max_pos_oversample)
        for split in dataset.keys()
    })
    tokenized = DatasetDict({
        split: sentence_ds[split].map(
            lambda batch: tokenize_sentence_batch(batch, tokenizer),
            batched=True,
            remove_columns=["sentence"],
            desc=f"[Extractive] 문장 토크나이징 ({split})",
        )
        for split in sentence_ds.keys()
    })

    # 3. Setup Trainer (Evaluation Mode)
    robust_weight = max(config.positive_class_weight, 4.0)
    trainer = RobustWeightedTrainer(
        pos_weight_value=robust_weight,
        model=model,
        args=TrainingArguments(
            output_dir=str(output_dir),
            per_device_eval_batch_size=config.per_device_train_batch_size,
            report_to=["none"],
            remove_unused_columns=True
        ),
        eval_dataset=tokenized["val"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_extractive_metrics_robust,
    )

    # 4. Evaluate
    metrics = trainer.evaluate()
    print(">>> Extractive Model (Loaded) 평가 결과:", metrics)

    # 5. Predict & Save
    predicted_dataset = build_robust_predicted_dataset(dataset, tokenized, trainer, top_k=3)
    PREDICTED_EXTRACTIVE_DATASETS[config.experiment_name] = predicted_dataset
    print(">>> Extractive Prediction 완료. 데이터셋이 메모리에 저장되었습니다.")

    return trainer, metrics, predicted_dataset

def train_robust_generative(config: ExperimentConfig):
    print("=== [Step 2] Robust Generative Model 학습 시작 ===")
    if config.experiment_name not in PREDICTED_EXTRACTIVE_DATASETS:
        print("Warning: 추출 요약 결과(Predicted Dataset)를 찾을 수 없습니다. eval_and_continue_extractive를 먼저 실행하세요.")
        return None

    dataset = PREDICTED_EXTRACTIVE_DATASETS[config.experiment_name]
    model, gen_tokenizer = build_generative_tokenizer(config)
    tokenized = tokenize_generative_dataset(dataset, config, gen_tokenizer)
    output_dir = resolve_experiment_dir(config, f"generative_{config.mode}")

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir), learning_rate=config.learning_rate, per_device_train_batch_size=config.per_device_train_batch_size, per_device_eval_batch_size=config.per_device_train_batch_size, num_train_epochs=config.num_train_epochs, gradient_accumulation_steps=config.gradient_accumulation_steps, predict_with_generate=True, generation_max_length=config.generation_max_length, eval_strategy="steps", save_strategy="steps", logging_steps=50, save_steps=200, eval_steps=200, save_total_limit=config.save_total_limit, report_to=["none"]
    )
    data_collator = DataCollatorForSeq2Seq(tokenizer=gen_tokenizer, model=model)
    compute_metrics = build_rouge_fn(gen_tokenizer)
    trainer = Seq2SeqTrainer(
        model=model, args=training_args, train_dataset=tokenized.get("train"), eval_dataset=tokenized.get("val"), tokenizer=gen_tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
    )

    resume_ckpt = None
    if config.resume:
        resume_ckpt = latest_checkpoint_path(output_dir)
        if resume_ckpt: print(f">>> [Generative] 체크포인트 발견: {resume_ckpt}. 학습을 재개합니다.")
        else: print(">>> [Generative] 체크포인트 없음. 처음부터 학습을 시작합니다.")

    trainer.train(resume_from_checkpoint=resume_ckpt)
    eval_metrics = trainer.evaluate(max_length=config.generation_max_length)
    log_metrics_to_csv(output_dir / "metrics.csv", eval_metrics)
    plot_training_history(trainer, output_dir / "history.png")
    print(">>> [Generative] 최종 평가 결과:", eval_metrics)
    return {"trainer": trainer, "metrics": eval_metrics, "preview": preview_generation_samples(trainer, gen_tokenizer, dataset, tokenized)}

# --- Execution Config ---
robust_hybrid_config = ExperimentConfig(
    mode="hybrid",
    dataset="news",
    experiment_name="news_robust_hybrid",
    num_train_epochs=1,
    positive_class_weight=4.0
)



In [32]:
# --- 실행 명령 ---
# 1.5. 중단된 Extractive 모델 평가 및 예측 데이터 생성 (학습 건너뛰기)
ext_trainer, ext_metrics, predicted_ds = eval_and_continue_extractive(robust_hybrid_config)

=== [Step 1.5] 중단된 Extractive 모델 평가 및 예측 데이터 생성 (Skip Training) ===
>>> 체크포인트에서 모델 로드 중: /content/drive/MyDrive/data/runs/news_robust_hybrid/robust_extractive/checkpoint-4000


[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/24324 [00:00<?, ? examples/s]

[hybrid] 추출 요약 필드 생성 (news):   0%|          | 0/3004 [00:00<?, ? examples/s]

[Extractive] 문장 토크나이징 (train):   0%|          | 0/291880 [00:00<?, ? examples/s]

[Extractive] 문장 토크나이징 (val):   0%|          | 0/18024 [00:00<?, ? examples/s]

/tmp/ipython-input-1447242197.py:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `RobustWeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)


>>> Extractive Model (Loaded) 평가 결과: {'eval_loss': 0.6368010640144348, 'eval_model_preparation_time': 0.0026, 'eval_accuracy': 0.6603972481136263, 'eval_precision': 0.6198491004062682, 'eval_recall': 0.829560585885486, 'eval_f1': 0.7095335263132919, 'eval_runtime': 206.1336, 'eval_samples_per_second': 87.438, 'eval_steps_per_second': 43.719}
[train] 예측 수행 중...
[train] Top-3 기반 추출 요약 생성 완료
[val] 예측 수행 중...


[val] Top-3 기반 추출 요약 생성 완료
>>> Extractive Prediction 완료. 데이터셋이 메모리에 저장되었습니다.


In [33]:
# 2. 생성 요약 학습
gen_result = train_robust_generative(robust_hybrid_config)
gen_result["preview"]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


=== [Step 2] Robust Generative Model 학습 시작 ===


You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BartTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


[hybrid] 생성 토크나이징 (news-train):   0%|          | 0/24324 [00:00<?, ? examples/s]

[hybrid] 생성 토크나이징 (news-val):   0%|          | 0/3004 [00:00<?, ? examples/s]

/tmp/ipython-input-1447242197.py:253: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


>>> [Generative] 체크포인트 없음. 처음부터 학습을 시작합니다.


Step,Training Loss,Validation Loss


OverflowError: out of range integral type conversion attempted